In [1]:
import os
import json
import argparse

In [2]:
from typing import List, Optional

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
from langchain_community.vectorstores.faiss import FAISS

In [5]:
from langchain.docstore.document import Document

In [6]:
class VectorSearchTester:
    """Класс для тестирования поиска по векторному индексу"""

In [8]:
    def __init__(self, 
                 index_path: str = "faiss_index_multi",
                 model_name: str = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"):
        """
        Инициализация тестера
        
        Args:
            index_path: Путь к сохраненному индексу
            model_name: Название модели эмбеддингов
        """
        self.index_path = index_path
        self.model_name = model_name
        
        # Загрузка модели эмбеддингов
        print(f"Загрузка модели эмбеддингов: {model_name}")
        self.embeddings = HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embeddings': True}
        )
        
        # Загрузка индекса
        self.load_index()

In [9]:
    def load_index(self):
        """Загрузка векторного индекса с диска"""
        if not os.path.exists(self.index_path):
            raise FileNotFoundError(f"Индекс не найден: {self.index_path}")
        
        print(f"Загрузка индекса из {self.index_path}")
        self.vectorstore = FAISS.load_local(
            self.index_path, 
            self.embeddings,
            allow_dangerous_deserialization=True
        )
        print("Индекс успешно загружен")

In [10]:
    def search(self, query: str, k: int = 5, threshold: float = 0.7) -> List:
        """
        Поиск по векторному индексу
        
        Args:
            query: Поисковый запрос
            k: Количество результатов
            threshold: Порог релевантности (0-1)
        
        Returns:
            Список найденных документов с метриками
        """
        # Поиск с оценками
        results = self.vectorstore.similarity_search_with_score(query, k=k)
        
        # Фильтрация по порогу релевантности
        filtered_results = []
        for doc, score in results:
            relevance = 1 - score  # Преобразование расстояния в релевантность
            if relevance >= threshold:
                filtered_results.append({
                    'document': doc,
                    'relevance': relevance,
                    'score': score
                })
        
        return filtered_results

In [11]:
    def display_results(self, query: str, results: List, verbose: bool = True):
        """
        Отображение результатов поиска
        
        Args:
            query: Исходный запрос
            results: Результаты поиска
            verbose: Показывать полный текст чанков
        """
        print(f"\n{'=' * 60}")
        print(f"ЗАПРОС: {query}")
        print(f"{'=' * 60}")
        
        if not results:
            print("❌ Релевантные документы не найдены")
            return
        
        print(f"✅ Найдено релевантных чанков: {len(results)}\n")
        
        for i, result in enumerate(results, 1):
            doc = result['document']
            relevance = result['relevance']
            
            print(f"{i}. 📄 Документ: {doc.metadata.get('filename', 'Unknown')}")
            print(f"   📊 Релевантность: {relevance:.2%}")
            print(f"   🔖 Чанк ID: {doc.metadata.get('chunk_id', 'N/A')}")
            print(f"   📍 Источник: {doc.metadata.get('source', 'N/A')}")
            
            if verbose:
                # Показываем первые 300 символов текста
                text_preview = doc.page_content[:300]
                if len(doc.page_content) > 300:
                    text_preview += "..."
                print(f"   📝 Текст:\n      {text_preview}\n")
            
            print("-" * 60)

In [12]:
    def test_golden_queries(self):
        """Тестирование на золотом наборе вопросов"""
        golden_queries = [
            {
                "query": "Кто такой Anthony Gibson?",
                "expected_doc": "Anthony_Gibson.txt",
                "description": "Вопрос о лесном духе Блуде"
            },
            {
                "query": "Что такое Dawn Wells?",
                "expected_doc": "Dawn_Wells.txt",
                "description": "Вопрос о неупокоенной душе"
            },
            {
                "query": "Расскажи про David Garcia",
                "expected_doc": "David_Garcia.txt",
                "description": "Вопрос о злыднях"
            },
            {
                "query": "Какие есть лесные духи?",
                "expected_doc": None,
                "description": "Общий вопрос о лесных духах"
            },
            {
                "query": "Как защититься от злых духов?",
                "expected_doc": None,
                "description": "Вопрос о защите от духов"
            }
        ]
        
        print("\n" + "=" * 60)
        print("ТЕСТИРОВАНИЕ НА ЗОЛОТОМ НАБОРЕ ВОПРОСОВ")
        print("=" * 60)
        
        results_summary = []
        
        for test_case in golden_queries:
            query = test_case["query"]
            expected = test_case["expected_doc"]
            description = test_case["description"]
            
            print(f"\n🔍 Тест: {description}")
            print(f"   Запрос: {query}")
            print(f"   Ожидается: {expected if expected else 'Любой релевантный документ'}")
            
            results = self.search(query, k=3, threshold=0.5)
            
            if results:
                top_result = results[0]['document']
                found_doc = top_result.metadata.get('filename', 'Unknown')
                relevance = results[0]['relevance']
                
                if expected:
                    # Проверка конкретного документа
                    success = found_doc == expected
                else:
                    # Любой релевантный результат считается успехом
                    success = True
                
                status = "✅ УСПЕХ" if success else "⚠️  НЕТОЧНО"
                print(f"   Результат: {status}")
                print(f"   Найден: {found_doc} (релевантность: {relevance:.2%})")
                
                results_summary.append({
                    'query': query,
                    'success': success,
                    'found': found_doc,
                    'relevance': relevance
                })
            else:
                print(f"   Результат: ❌ НЕ НАЙДЕНО")
                results_summary.append({
                    'query': query,
                    'success': False,
                    'found': None,
                    'relevance': 0
                })
        
        # Итоговая статистика
        print("\n" + "=" * 60)
        print("ИТОГОВАЯ СТАТИСТИКА")
        print("=" * 60)
        
        successful = sum(1 for r in results_summary if r['success'])
        total = len(results_summary)
        accuracy = successful / total * 100
        
        print(f"Успешных тестов: {successful}/{total} ({accuracy:.1f}%)")
        print(f"Средняя релевантность: {sum(r['relevance'] for r in results_summary) / total:.2%}")
        
        return results_summary

In [13]:
    def interactive_search(self):
        """Интерактивный режим поиска"""
        print("\n" + "=" * 60)
        print("ИНТЕРАКТИВНЫЙ ПОИСК")
        print("=" * 60)
        print("Введите запросы для поиска (или 'выход' для завершения)")
        print("Формат: <запрос> [количество результатов]")
        print("Пример: Кто такой лесной дух? 3")
        
        while True:
            try:
                user_input = input("\n🔍 Запрос: ").strip()
                
                if user_input.lower() in ['выход', 'exit', 'quit', 'q']:
                    print("Завершение работы...")
                    break
                
                if not user_input:
                    continue
                
                # Парсинг запроса и количества результатов
                parts = user_input.rsplit(' ', 1)
                if len(parts) == 2 and parts[1].isdigit():
                    query = parts[0]
                    k = int(parts[1])
                else:
                    query = user_input
                    k = 3
                
                # Выполнение поиска
                results = self.search(query, k=k, threshold=0.3)
                self.display_results(query, results)
                
            except KeyboardInterrupt:
                print("\n\nПрервано пользователем")
                break
            except Exception as e:
                print(f"❌ Ошибка: {e}")

In [14]:
def main():
    """Главная функция"""
    parser = argparse.ArgumentParser(description='Тестирование векторного поиска')
    parser.add_argument('--index', default='faiss_index_multi', help='Путь к индексу')
    parser.add_argument('--model', default='sentence-transformers/paraphrase-multilingual-mpnet-base-v2', 
                        help='Модель эмбеддингов')
    parser.add_argument('--query', help='Одиночный запрос для поиска')
    parser.add_argument('--k', type=int, default=3, help='Количество результатов')
    parser.add_argument('--test', action='store_true', 
                        help='Запустить тестирование на золотом наборе')
    parser.add_argument('--interactive', action='store_true', 
                        help='Интерактивный режим')
    
    args = parser.parse_args()
    
    # Создание тестера
    tester = VectorSearchTester(args.index, args.model)
    
    if args.test:
        # Тестирование на золотом наборе
        tester.test_golden_queries()
    elif args.query:
        # Одиночный запрос
        results = tester.search(args.query, k=args.k)
        tester.display_results(args.query, results)
    elif args.interactive:
        # Интерактивный режим
        tester.interactive_search()
    else:
        # По умолчанию - интерактивный режим
        tester.interactive_search()

In [15]:
if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] [--index INDEX] [--model MODEL]
                             [--query QUERY] [--k K] [--test] [--interactive]
ipykernel_launcher.py: error: unrecognized arguments: -f /Users/vladimir/Library/Jupyter/runtime/kernel-bd299804-3ed8-4a76-ab8d-e472cea1ab55.json


SystemExit: 2

/Users/vladimir/anaconda3/envs/architecture-pro-rag/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
